In [0]:
"""
The script which calculates intermediate features related to transactions.
It only depends on source etl intermediates.
"""
import sys
sys.path.append('..')
sys.path.append('../..')
import lib_dna_member.generate_population as gp
import lib_dna_member.job_manager as managers
import lib_dna_member.transaction_features as features
import lib_dna_member.utils as utils
from lib_dna_member.s3 import member_dna_input_data_validator
from databricks.feature_engineering import FeatureEngineeringClient

In [0]:
%run ../../config/utils

In [0]:

def generate_transaction(job):
    """
    Generate transaction based variables for the given population
    Parameters:
        job (object): Job Manager object based on the current config file

    Returns:
        dna (pyspark.sql.DataFrame): Transaction data of the given population
    """
    dna = job.tables["population"]
    orig_cols = dna.columns
    dna = features.feature_grouped_tender_spend_nw(job, dna)
    dna = features.feature_member_category(
        job,
        dna,
        ["MEN", "MENS", "MEN''S"],
        ["WOMEN", "WOMENS", "WOMEN''S"],
        ["AH4_DESC", "AH5_DESC", "AH6_DESC"],
        "MEN",
        [52],
    )
    dna = features.feature_member_category(
        job,
        dna,
        ["WOMEN", "WOMENS", "WOMEN''S"],
        ["MEN", "MENS", "MEN''S"],
        ["AH4_DESC", "AH5_DESC", "AH6_DESC"],
        "WOMEN",
        [52],
    )
    dna = features.feature_member_category(
        job,
        dna,
        ["PET", "PETS", "PET''S"],
        ["FUNERAL", "URN"],
        ["AH4_DESC", "AH5_DESC", "AH6_DESC"],
        "PET",
        [52],
    )
    dna = features.feature_member_category(
        job,
        dna,
        [
            "CHILDREN",
            "CHILDREN''S",
            "KID",
            "KIDS",
            "KID''S",
            "GIRL",
            "GIRLS",
            "GIRL''S",
            "BOY",
            "BOYS",
            "BOY''S",
        ],
        ["NEWBORN/INFANT", "CEREAL"],
        ["AH4_DESC", "AH5_DESC", "AH6_DESC"],
        "CHILDREN",
        [52],
    )
    dna = features.feature_member_category(
        job,
        dna,
        [
            "BABY",
            "BABY''S",
            "BABIES",
            "INFANT",
            "DIAPERS",
            "NEWBORN",
            "INFANT",
        ],
        ["RIBS", "COTTON SWABS", "WIPES"],
        ["AH4_DESC", "AH5_DESC", "AH6_DESC"],
        "BABY",
        [52],
    )

    # dna = utils.cache_df(dna)

    dna = features.feature_member_basket_size(job, dna, 51)
    dna = features.feature_distinct_categories(job, dna)
    dna = features.feature_days_since_last_trip(job, dna)
    dna = features.feature_trip_intervals(job, dna, 52)

    # dna = utils.cache_df(dna)

    dna = dna.drop(
        *[
            col
            for col in orig_cols
            if col not in ["MBRSHP_SID", "FISCAL_WEEK_END"]
        ]
    )

    return dna



In [0]:

def main():
    job = managers.JobManager(spark, intermediate_all_tables_dict, member_dna_config_path)
    
    recency_lookback_duration = job.config["params"].get("recency_lookback_duration", {})

    skeleton_path = intermediate_all_tables_dict['skeleton']
    member_extended_path = intermediate_all_tables_dict['member_extended']
    payment_path = intermediate_all_tables_dict['payment_fiscal']
    detail_isnr_path = intermediate_all_tables_dict['detail_isnr_fiscal']
    detail_path = intermediate_all_tables_dict['detail_fiscal']
    club_path   = intermediate_all_tables_dict['club']
    tender_map_path = intermediate_all_tables_dict['tender_map']
    #fs_transaction_3 = intermediate_all_tables_dict["cubes_transaction_3"]


    member_dna_input_data_validator(
        skeleton_path,
        member_extended_path,
        payment_path,
        tender_map_path,
        detail_isnr_path,
        detail_path,
        club_path,
        recency_lookback_duration=recency_lookback_duration,
        spark=spark
    )   

    # member_dna_input_data_validator(
        # job,
        # recency_lookback_duration,
        # [
            #"skeleton_path",
            # "member_extended_path",
            # "payment_path",
            # "tender_map_path",
            # "detail_isnr_path",
            # "detail_path",
            # "club_path",
        # ],
    # )
    job.read_table("skeleton")
    job.read_table("member_extended")
    job.read_table("payment_fiscal")
    job.read_table("tender_map")
    job.read_table("detail_isnr_fiscal")
    job.read_table("detail_fiscal")
    job.read_table("club")

    job.tables["detail_isnr_fiscal"] = gp.apply_fw_date_range(
        job, job.tables["detail_isnr_fiscal"]
    )

    job.tables["detail_fiscal"] = gp.apply_fw_date_range(
        job, job.tables["detail_fiscal"]
    )

    job.tables["payment_fiscal"] = gp.apply_fw_date_range(
        job, job.tables["payment_fiscal"]
    )

    population = gp.generate_population(job)
    job.tables["population"] = population

    feature_population = gp.generate_population(job, "feature")
    job.tables["feature_population"] = feature_population

    features = generate_transaction(job)
    features = features.withColumn('MBRSHP_SID', f.coalesce('MBRSHP_SID', f.lit(-1)))
    
    spark.sql(f"DELETE FROM {fs_cubes_transaction_3}")

    fe = FeatureEngineeringClient()

    fe.write_table(
        name=fs_cubes_transaction_3,
        df=features,
        mode="merge"
    )



In [0]:

if __name__ == "__main__":
    main()